# Cybersecurity Recommendation System — Notebook 01
## Public data ingestion and exploratory data analysis (EDA)

**Goal:** fetch the official CISA Known Exploited Vulnerabilities (KEV) JSON feed, inspect its structure, build a tidy table, and summarize the data. This is a defensive research workflow; it does not test or exploit systems.

**Beginner workflow:** run the cells from top to bottom. Outputs are produced from the live feed when you run the notebook; this notebook does not assume or invent row counts or findings.

**Source:** [CISA KEV Catalog](https://www.cisa.gov/known-exploited-vulnerabilities-catalog) · [Official JSON feed](https://www.cisa.gov/sites/default/files/feeds/known_exploited_vulnerabilities.json)

**Important:** the catalog changes over time. Record the catalog version and download date for reproducibility. KEV entries indicate vulnerabilities CISA has identified as exploited in the wild; absence from KEV does not mean a vulnerability is harmless.

## 1. Import libraries
Pandas helps us work with tables. Requests downloads the public JSON feed. Python's `datetime` records when we collected it.

In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import json
import requests
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 50)
pd.set_option('display.max_colwidth', 120)
print('Libraries imported successfully.')

## 2. Download the official CISA KEV feed
This makes one read-only HTTP request to CISA's public JSON file. A timeout prevents the request from waiting forever. If the request fails, check your internet connection and try again later.

In [ ]:
KEV_URL = 'https://www.cisa.gov/sites/default/files/feeds/known_exploited_vulnerabilities.json'
response = requests.get(KEV_URL, timeout=30)
response.raise_for_status()  # Stop with an error if the server returned an unsuccessful status.
kev_json = response.json()

downloaded_at_utc = datetime.now(timezone.utc).isoformat()
print('Downloaded successfully at:', downloaded_at_utc)
print('Catalog version:', kev_json.get('catalogVersion', 'Not supplied'))
print('Catalog release date:', kev_json.get('dateReleased', 'Not supplied'))
print('Count stated by feed:', kev_json.get('count', 'Not supplied'))
print('Number of records in vulnerabilities array:', len(kev_json.get('vulnerabilities', [])))

## 3. Inspect the JSON structure
JSON is organized as key-value pairs. The feed contains catalog metadata and a `vulnerabilities` list. We inspect the available field names before choosing columns.

In [ ]:
print('Top-level keys:', list(kev_json.keys()))
records = kev_json.get('vulnerabilities', [])
if not isinstance(records, list):
    raise TypeError("Expected 'vulnerabilities' to be a list in the CISA JSON feed.")
print('Records loaded:', len(records))
if records:
    print('Fields in first record:', list(records[0].keys()))
    display(pd.json_normalize(records[:3]))
else:
    print('The feed contains no records in this response; stop and check the source before continuing.')

## 4. Convert the records into a DataFrame
A DataFrame is a spreadsheet-like table. We standardize column names and keep the source fields useful for defensive prioritization and later recommendation experiments. Missing fields are allowed; we do not invent values.

In [ ]:
df = pd.json_normalize(records)
df.columns = [str(column).strip() for column in df.columns]

# Rename the main identifier to a consistent name if the source field is present.
if 'cveID' in df.columns:
    df = df.rename(columns={'cveID': 'cve_id'})

print('Table shape (rows, columns):', df.shape)
display(df.head(5))

## 5. Basic quality checks
We check duplicated rows, duplicated CVE identifiers, and missing values. Missingness is reported rather than automatically filled because an empty field can have a specific meaning in the source.

In [ ]:
print('Exact duplicate rows:', int(df.duplicated().sum()))
if 'cve_id' in df.columns:
    print('Missing CVE IDs:', int(df['cve_id'].isna().sum()))
    print('Duplicate CVE IDs:', int(df['cve_id'].duplicated().sum()))

missing_summary = (df.isna().sum().rename('missing_count').to_frame())
missing_summary['missing_percent'] = (missing_summary['missing_count'] / max(len(df), 1) * 100).round(2)
display(missing_summary.sort_values('missing_count', ascending=False))

## 6. Parse date fields
The catalog usually includes `dateAdded` and `dueDate`. We parse these as dates so later analysis can group records by year or month. Unparseable or missing values become `NaT` (a missing datetime), not a guessed date.

In [ ]:
for date_column in ['dateAdded', 'dueDate']:
    if date_column in df.columns:
        df[date_column] = pd.to_datetime(df[date_column], errors='coerce')

date_columns_present = [c for c in ['dateAdded', 'dueDate'] if c in df.columns]
if date_columns_present:
    display(df[date_columns_present].describe(datetime_is_numeric=True) if hasattr(df[date_columns_present], 'describe') else df[date_columns_present].head())
    for column in date_columns_present:
        print(f'{column}: missing/unparseable = {int(df[column].isna().sum())}')
else:
    print('No expected date fields were present. Review the source schema before proceeding.')

## 7. Explore the catalog
The following tables summarize the data as downloaded. They do not imply that one vendor or product is inherently less secure; the counts depend on what CISA has added to this catalog.

In [ ]:
for column in ['vendorProject', 'product', 'knownRansomwareCampaignUse']:
    if column in df.columns:
        print(f'\nTop values for {column}:')
        display(df[column].fillna('Missing / not supplied').value_counts().head(10).rename_axis(column).to_frame('record_count'))
    else:
        print(f'Column not found in this feed version: {column}')

In [ ]:
if 'dateAdded' in df.columns:
    dated = df.dropna(subset=['dateAdded']).copy()
    if not dated.empty:
        dated['added_year'] = dated['dateAdded'].dt.year
        yearly_counts = dated.groupby('added_year').size().sort_index()
        ax = yearly_counts.plot(kind='bar', figsize=(10, 4), title='CISA KEV records by date added (current catalog snapshot)')
        ax.set_xlabel('Year')
        ax.set_ylabel('Number of records')
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()
        display(yearly_counts.rename('record_count').to_frame())
    else:
        print('No parseable dateAdded values are available to plot.')
else:
    print('dateAdded is not available; skipping the date chart.')

## 8. Save a local research snapshot
We save the original JSON and a CSV table to the Colab runtime. These files are temporary unless you download them or copy them to Google Drive. The repository's `.gitignore` intentionally excludes local data so a changing dataset is not accidentally committed.

In [ ]:
output_dir = Path('data')
output_dir.mkdir(parents=True, exist_ok=True)
snapshot_stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
json_path = output_dir / f'cisa_kev_{snapshot_stamp}.json'
csv_path = output_dir / f'cisa_kev_{snapshot_stamp}.csv'
metadata_path = output_dir / f'cisa_kev_{snapshot_stamp}_metadata.json'

json_path.write_text(json.dumps(kev_json, indent=2), encoding='utf-8')
df.to_csv(csv_path, index=False)
metadata = {
    'source_url': KEV_URL,
    'downloaded_at_utc': downloaded_at_utc,
    'catalog_version': kev_json.get('catalogVersion'),
    'date_released': kev_json.get('dateReleased'),
    'feed_count': kev_json.get('count'),
    'records_loaded': len(df),
    'columns': list(df.columns),
}
metadata_path.write_text(json.dumps(metadata, indent=2), encoding='utf-8')
print('Saved files:')
print(json_path)
print(csv_path)
print(metadata_path)

## 9. Write down your observations
After running the notebook, use the live outputs to fill in this short research log. Do not copy example numbers from elsewhere or report values before running the cells.

- Catalog version and download date:
- Number of records loaded:
- Most common vendors/products in this snapshot:
- Missing fields that may affect later analysis:
- Any duplicate rows or CVE IDs:
- Limitations or questions to investigate:

**Interpretation caution:** KEV is a curated catalog of known exploited vulnerabilities, not a complete inventory of every vulnerability and not a user-feedback dataset. Counts in this notebook are descriptive only; they are not model evaluation scores.

## 10. What comes next?
The next notebook will perform preprocessing in a reproducible way. Before training a recommendation model, we must define the recommendation task and document how relevance labels will be obtained. Public catalog entries alone do not provide real user ratings, so we must not fabricate ratings or claim that a synthetic proxy is genuine user feedback.

### Source links
- CISA KEV catalog: https://www.cisa.gov/known-exploited-vulnerabilities-catalog
- Official CISA JSON feed: https://www.cisa.gov/sites/default/files/feeds/known_exploited_vulnerabilities.json
- NVD (possible additional source for a later notebook): https://nvd.nist.gov/developers/vulnerabilities
